In [1]:
import os
import gc

import numpy as np
import pandas as pd
import seaborn as sns
import missingno as msno
sns.set(style='darkgrid')
import matplotlib.pyplot as plt
from pathlib import Path
import joblib

print("Working directory:", Path.cwd())

# include old CIC IDS data
data_list = [pd.read_csv(csv) for csv in Path("../fixed_data/").iterdir()]
data = pd.concat(data_list)
data_len = len(data)
print("concated CIC IDS data")

# include new generated data
data2_attack_list = [pd.read_csv(csv) for csv in Path("../scale_data/attacks/").iterdir()]
data2_attack = pd.concat(data2_attack_list)
data2_attack['Label'] = 'ATTACK'

data2_benign_list = [pd.read_csv(csv) for csv in Path("../scale_data/benign/").iterdir()]
data2_benign = pd.concat(data2_benign_list)
data2_benign['Label'] = 'BENIGN'

data2 = pd.concat([data2_attack, data2_benign])
print("concated our data")

del data2_attack
del data2_benign
gc.collect()

# scale data in case we don't have enough
"""
original_positions = np.linspace(
    0, len(data2) - 1, data_len // 4, dtype=int
)
data2_large = data2.iloc[original_positions].reset_index(drop=True)
"""
data2_large = data2.copy()
print(data.shape)
print(data2_large.shape)

del data2
gc.collect()

data_half = data.sample(frac=0.5).reset_index(drop=True)
print("sampled first half of data")
data2_half = data2_large.sample(frac=0.5).reset_index(drop=True)
print("sampled second half of data")

del data2_large
del data
gc.collect()

data = pd.concat([data_half, data2_half], join='inner', ignore_index=True)
print(data.shape)

# Deleting dataframes after concating to save memory
for d in data_list: del d
del data2_half
del data_half
gc.collect()

# Renaming the columns by removing leading/trailing whitespace
col_names = {col: col.strip() for col in data.columns}
data.rename(columns = col_names, inplace = True)
data.columns

print("done entering data")

Working directory: /home/weluv/CMPE-272/src
concated CIC IDS data
concated our data
(2100816, 85)
(12498, 95)
sampled first half of data
sampled second half of data
(1056657, 85)
done entering data


In [2]:
for col in data.select_dtypes(include='float').columns:
    s = data[col]
    
    # 1. Skip columns containing infinite values (inf cannot become an int)
    if np.isinf(s).any():
        continue
        
    # 2. Check if the non-null values are all whole numbers
    non_null = s.dropna()
    if not non_null.empty and non_null.mod(1).eq(0).all():
        # 3. Use 'Int64' (nullable) if there are NaNs, otherwise standard 'int64'
        if s.isna().any():
            data[col] = s.astype('Int64')
        else:
            data[col] = s.astype('int64')

data.drop(columns=['[ZoneTransfer]'], errors='ignore', inplace=True)
data['Label'] = data['Label'].astype('object')
data['Label'] = data['Label'].apply(lambda x: 'BENIGN' if x == 'BENIGN' else 'ATTACK')

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1056657 entries, 0 to 1056656
Data columns (total 84 columns):
 #   Column                      Non-Null Count    Dtype  
---  ------                      --------------    -----  
 0   Flow ID                     1056655 non-null  object 
 1   Src IP                      1056655 non-null  object 
 2   Src Port                    1056655 non-null  Int64  
 3   Dst IP                      1056655 non-null  object 
 4   Dst Port                    1056655 non-null  Int64  
 5   Protocol                    1056655 non-null  Int64  
 6   Timestamp                   1056655 non-null  object 
 7   Flow Duration               1056655 non-null  Int64  
 8   Total Fwd Packet            1056655 non-null  Int64  
 9   Total Bwd packets           1056655 non-null  Int64  
 10  Total Length of Fwd Packet  1056655 non-null  Int64  
 11  Total Length of Bwd Packet  1056655 non-null  Int64  
 12  Fwd Packet Length Max       1056655 non-null  Int64  
 1

In [3]:
pd.options.display.max_columns = 90
data

,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,Total Length of Fwd Packet,Total Length of Bwd Packet,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Packet Length Min,Packet Length Max,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWR Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Fwd Segment Size Avg,Bwd Segment Size Avg,Fwd Bytes/Bulk Avg,Fwd Packet/Bulk Avg,Fwd Bulk Rate Avg,Bwd Bytes/Bulk Avg,Bwd Packet/Bulk Avg,Bwd Bulk Rate Avg,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,FWD Init Win Bytes,Bwd Init Win Bytes,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,192.168.10.19-192.168.10.3-17203-53-17,192.168.10.19,17203,192.168.10.3,53,17,06/07/2017 06:57:55 PM,226,2,2,78,184,39,39,39.000000,0.000000,92,92,92.000000,0.000000,1.159292e+06,17699.115044,7.533333e+01,1.287491e+02,224,1,1,1.000000e+00,0.000000e+00,1,1,1,1.000000e+00,0.000000e+00,1,1,0,0,0,0,16,16,8849.557522,8849.557522,39,92,65.500000,30.599564,9.363333e+02,0,0,0,0,0,0,0,0,1.000000,65.500000,39.000000,92.000000,0,0,0,0,0,0,0.0,19.0,0.0,46.0,0,0,1,8,0.000000e+00,0.000000e+00,0,0,0.000000e+00,0.000000e+00,0,0,BENIGN
1,192.168.10.14-192.168.10.3-62191-53-17,192.168.10.14,62191,192.168.10.3,53,17,06/07/2017 07:44:46 PM,24261,2,2,74,194,37,37,37.000000,0.000000,97,97,97.000000,0.000000,1.104654e+04,164.873666,8.087000e+03,1.396468e+04,24212,1,1,1.000000e+00,0.000000e+00,1,1,48,4.800000e+01,0.000000e+00,48,48,0,0,0,0,16,16,82.436833,82.436833,37,97,67.000000,34.641016,1.200000e+03,0,0,0,0,0,0,0,0,1.000000,67.000000,37.000000,97.000000,0,0,0,0,0,0,0.0,18.0,0.0,48.0,0,0,1,8,0.000000e+00,0.000000e+00,0,0,0.000000e+00,0.000000e+00,0,0,BENIGN
2,192.168.10.14-204.154.111.110-50262-443-6,192.168.10.14,50262,204.154.111.110,443,6,06/07/2017 08:05:26 PM,5145210,10,10,1855,6019,1527,0,185.500000,476.597489,1955,0,601.900000,788.373494,1.530355e+03,3.887111,2.708005e+05,1.145856e+06,5002254,3,5112348,5.680387e+05,1.662943e+06,5002338,3,5112451,5.680501e+05,1.662927e+06,5002290,3,3,4,0,0,212,204,1.943555,1.943555,0,1955,393.700000,669.054565,4.476340e+05,2,2,0,7,19,0,0,0,1.000000,393.700000,185.500000,601.900000,0,0,0,0,0,0,0.0,92.0,0.0,300.0,8192,32120,3,20,1.098870e+05,0.000000e+00,109887,109887,5.002254e+06,0.000000e+00,5002254,5002254,BENIGN
3,192.168.10.14-173.241.242.143-52368-443-6,192.168.10.14,52368,173.241.242.143,443,6,04/07/2017 03:33:44 PM,37667855,11,18,843,3940,527,0,76.636364,162.587375,1460,0,218.888889,464.782921,1.269783e+02,0.769887,1.345281e+06,3.148347e+06,10002729,2,30141369,3.014137e+06,4.820800e+06,10038495,116,37632027,2.213649e+06,3.844091e+06,10038676,2,3,10,0,0,232,396,0.292026,0.477861,0,1460,164.931034,381.486840,1.455322e+05,8,2,1,13,27,0,0,0,1.636364,164.931034,76.636364,218.888889,0,0,0,124,4,70,0.0,29.0,0.0,135.0,8192,62,5,20,7.092833e+04,6.074626e+04,141072,35766,9.976115e+06,4.527370e+04,10002729,9923840,BENIGN
4,192.168.10.8-192.168.10.19-59793-5060-6,192.168.10.8,59793,192.168.10.19,5060,6,06/07/2017 08:40:30 PM,20,1,1,0,0,0,0,0.000000,0.000000,0,0,0.000000,0.000000,0.000000e+00,100000.000000,2.000000e+01,0.000000e+00,20,20,0,0.000000e+00,0.000000e+00,0,0,0,0.000000e+00,0.000000e+00,0,0,0,0,0,0,24,20,50000.000000,50000.000000,0,0,0.000000,0.000000,0.000000e+00,0,1,1,0,1,0,0,0,1.000000,0.000000,0.000000,0.000000,0,

In [4]:
# data cleaning

# data.drop_duplicates(inplace = True)
data = data.dropna()
data.replace([np.inf, -np.inf], np.nan, inplace = True)

# Filling missing values with median
med_flow_bytes = data['Flow Bytes/s'].median()
med_flow_packets = data['Flow Packets/s'].median()
data['Flow Bytes/s'].fillna(med_flow_bytes, inplace = True)
data['Flow Packets/s'].fillna(med_flow_packets, inplace = True)

print("done")
data.info()

/tmp/ipykernel_1931/3518974408.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.replace([np.inf, -np.inf], np.nan, inplace = True)
/tmp/ipykernel_1931/3518974408.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['Flow Bytes/s'].fillna(med_flow_bytes, inplace = True)
/tmp/ipykernel_1931/3518974408.py:10: SettingWithCopyWarning: 

done
<class 'pandas.core.frame.DataFrame'>
Index: 1056261 entries, 0 to 1056656
Data columns (total 84 columns):
 #   Column                      Non-Null Count    Dtype  
---  ------                      --------------    -----  
 0   Flow ID                     1056261 non-null  object 
 1   Src IP                      1056261 non-null  object 
 2   Src Port                    1056261 non-null  Int64  
 3   Dst IP                      1056261 non-null  object 
 4   Dst Port                    1056261 non-null  Int64  
 5   Protocol                    1056261 non-null  Int64  
 6   Timestamp                   1056261 non-null  object 
 7   Flow Duration               1056261 non-null  Int64  
 8   Total Fwd Packet            1056261 non-null  Int64  
 9   Total Bwd packets           1056261 non-null  Int64  
 10  Total Length of Fwd Packet  1056261 non-null  Int64  
 11  Total Length of Bwd Packet  1056261 non-null  Int64  
 12  Fwd Packet Length Max       1056261 non-null  Int64  
 1

In [5]:
# Creating a dictionary that maps each label to its attack type
attack_map = {
    'BENIGN': 'BENIGN',
    'DDoS': 'DDoS',
    'DoS Hulk': 'DoS',
    'DoS GoldenEye': 'DoS',
    'DoS slowloris': 'DoS',
    'DoS Slowhttptest': 'DoS',
    'PortScan': 'Port Scan',
    'FTP-Patator': 'Brute Force',
    'SSH-Patator': 'Brute Force',
    'Bot': 'Bot',
    'Web Attack � Brute Force': 'Web Attack',
    'Web Attack � XSS': 'Web Attack',
    'Web Attack � Sql Injection': 'Web Attack',
    'Infiltration': 'Infiltration',
    'Heartbleed': 'Heartbleed',
    'ATTACK': 'ATTACK'
}

# Creating a new column 'Attack Type' in the DataFrame based on the attack_map dictionary
if 'Label' in data.columns:
    data['Attack Type'] = data['Label'].map(attack_map)

    data.drop('Label', axis = 1, inplace = True)

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
data['Attack Number'] = le.fit_transform(data['Attack Type'])

print("done mapping data")

/tmp/ipykernel_1931/2130954140.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Attack Type'] = data['Label'].map(attack_map)
/tmp/ipykernel_1931/2130954140.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.drop('Label', axis = 1, inplace = True)


done mapping data


/tmp/ipykernel_1931/2130954140.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Attack Number'] = le.fit_transform(data['Attack Type'])


In [6]:
# data processing

for col in data.columns:
    col_type = data[col].dtype
    if col_type != object:
        c_min = data[col].min()
        c_max = data[col].max()
        # Downcasting float64 to float32
        if str(col_type).find('float') >= 0 and c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
            data[col] = data[col].astype(np.float32)

        # Downcasting int64 to int32
        elif str(col_type).find('int') >= 0 and c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
            data[col] = data[col].astype(np.int32)
            
data.info()

/tmp/ipykernel_1931/839654246.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[col] = data[col].astype(np.float32)
/tmp/ipykernel_1931/839654246.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[col] = data[col].astype(np.float32)
/tmp/ipykernel_1931/839654246.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/st

<class 'pandas.core.frame.DataFrame'>
Index: 1056261 entries, 0 to 1056656
Data columns (total 85 columns):
 #   Column                      Non-Null Count    Dtype  
---  ------                      --------------    -----  
 0   Flow ID                     1056261 non-null  object 
 1   Src IP                      1056261 non-null  object 
 2   Src Port                    1056261 non-null  Int64  
 3   Dst IP                      1056261 non-null  object 
 4   Dst Port                    1056261 non-null  Int64  
 5   Protocol                    1056261 non-null  Int64  
 6   Timestamp                   1056261 non-null  object 
 7   Flow Duration               1056261 non-null  Int64  
 8   Total Fwd Packet            1056261 non-null  Int64  
 9   Total Bwd packets           1056261 non-null  Int64  
 10  Total Length of Fwd Packet  1056261 non-null  Int64  
 11  Total Length of Bwd Packet  1056261 non-null  Int64  
 12  Fwd Packet Length Max       1056261 non-null  Int64  
 13  Fw

In [ ]:
# 

In [7]:
# Dropping columns with only one unique value
num_unique = data.nunique()
one_variable = num_unique[num_unique == 1]
not_one_variable = num_unique[num_unique > 1].index

dropped_cols = one_variable.index
data = data[not_one_variable]

print('Dropped columns:')
dropped_cols

Dropped columns:


Index([], dtype='object')

In [9]:
# Columns after removing non variant columns
data.columns

Index(['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol',
       'Timestamp', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets',
       'Total Length of Fwd Packet', 'Total Length of Bwd Packet',
       'Fwd Packet Length Max', 'Fwd Packet Length Min',
       'Fwd Packet Length Mean', 'Fwd Packet Length Std',
       'Bwd Packet Length Max', 'Bwd Packet Length Min',
       'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
       'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
       'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std',
       'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean',
       'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags',
       'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Packet Length Min', 'Packet Length Max', 'Packet Length Mean',
       'Packet Length Std', 'Packet Len

In [10]:
# drop columns that would leak data
columns_to_drop = ['Flow ID', 'Src IP', 'Dst IP', 'Src Port', 'Dst Port', 'Timestamp']
data = data.drop(columns=columns_to_drop, errors='ignore')

# Keep only TCP flows
print("Before TCP filter:", len(data))
print(data['Protocol'].value_counts())

data = data[data['Protocol'] == 6].copy()

print("After TCP filter:", len(data))
print(data['Attack Type'].value_counts())   # check each family survived

# drop more faulty columns
data = data.drop(columns=['Protocol', 'FWD Init Win Bytes', 'Bwd Init Win Bytes',
                          'Fwd Seg Size Min', 'Packet Length Min',
                          'Fwd Packet Length Min', 'Bwd Packet Length Min'],
                 errors='ignore')
data.columns

Before TCP filter: 1056261
Protocol
6     555476
17    499916
0        816
1         53
Name: count, dtype: Int64
After TCP filter: 555476
Attack Type
BENIGN    331086
ATTACK    224390
Name: count, dtype: int64


Index(['Flow Duration', 'Total Fwd Packet', 'Total Bwd packets',
       'Total Length of Fwd Packet', 'Total Length of Bwd Packet',
       'Fwd Packet Length Max', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
       'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
       'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std',
       'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean',
       'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags',
       'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Packet Length Max', 'Packet Length Mean', 'Packet Length Std',
       'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count',
       'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count',
       'CWR Flag Count', 'ECE Flag 

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Config
CV_FOLDS = 5
SEED = 42

# Data prep. 
# replace y with the commented out stuff if we want multi-class
"""
y = le.fit_transform(data["Attack Type"].astype(str))
""" 
y = (data["Attack Type"].astype(str).str.upper() != "BENIGN").astype(np.int8).to_numpy()
X = data.drop(columns=["Attack Type", "Attack Number"]).to_numpy(dtype=np.float32)
feature_names = [c for c in data.columns if c not in ("Attack Type", "Attack Number")]
attack_type = data["Attack Type"].to_numpy()   # kept for per-family breakdown
# del data
gc.collect()
 
X_train, X_test, y_train, y_test, _, type_test = train_test_split(
    X, y, attack_type, test_size=0.2, stratify=y, random_state=SEED
)
print(type(X_train))
del X, y
gc.collect()
print(f"Train {X_train.shape}, test {X_test.shape}, attack rate {y_train.mean():.3f}")

# Model: no scaler / PCA / SMOTE needed
model = HistGradientBoostingClassifier(
    max_iter=75,
    learning_rate=0.1,
    class_weight="balanced",   # handles imbalance without synthetic samples
    early_stopping=True,       # internal validation split stops training when it plateaus
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=SEED,
    verbose=1
)
 
# Stratified k-fold cross-validation
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
res = cross_validate(
    model, X_train, y_train, cv=cv,
    scoring=["accuracy", "f1", "precision", "recall"],
    n_jobs=1,   # each fit already uses all cores; parallel folds would multiply memory
)
for m in ["accuracy", "f1", "precision", "recall"]:
    s = res[f"test_{m}"]
    print(f"CV {m:<9}: {np.round(s, 4)}  mean={s.mean():.4f} (+/- {s.std():.4f})")
 
# Final model on the full training set, scored on the untouched test set
model.fit(X_train, y_train)
print(f"\nStopped at {model.n_iter_} iterations")

# dump columns for use later
joblib.dump(model, "hgb_model.joblib")
joblib.dump(le, "label_encoder.joblib")
joblib.dump(feature_names, "feature_columns.joblib")
 
y_pred = model.predict(X_test)
print(f"Test accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Test F1:       {f1_score(y_test, y_pred):.4f}")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["Benign", "Attack"], digits=4))

# Recall per attack family (shows which attacks are hardest to catch)
print("\nDetection rate by original attack type (test set):")
for t in np.unique(type_test):
    mask = type_test == t
    if str(t).upper() == "BENIGN":
        print(f"  {t:<30} n={mask.sum():>8}  correctly benign: {(y_pred[mask] == 0).mean():.4f}")
    else:
        print(f"  {t:<30} n={mask.sum():>8}  detected:         {(y_pred[mask] == 1).mean():.4f}")

<class 'numpy.ndarray'>
Train (444380, 70), test (111096, 70), attack rate 0.404
Binning 0.179 GB of training data: 2.340 s
Binning 0.020 GB of validation data: 0.115 s
Fitting gradient boosted rounds:


In [13]:
from sklearn.metrics import roc_auc_score
X_train_df = pd.DataFrame(X_train, columns=feature_names)
aucs = {c: roc_auc_score(y_train, X_train_df[c].fillna(0)) for c in X_train_df.columns}
s = pd.Series(aucs).sub(0.5).abs().sort_values(ascending=False)
print(s.head(15))

RST Flag Count                0.409254
FIN Flag Count                0.394665
Flow Bytes/s                  0.307964
Fwd Packet Length Std         0.305066
Bwd PSH Flags                 0.276838
Total Length of Bwd Packet    0.274373
Packet Length Max             0.272759
Packet Length Mean            0.271040
Average Packet Size           0.271040
Packet Length Variance        0.269643
Packet Length Std             0.269643
Bwd Packet Length Std         0.263944
Bwd Packet Length Max         0.263478
Down/Up Ratio                 0.261551
Bwd Packet Length Mean        0.260228
dtype: float64


In [14]:
cols = ['RST Flag Count','Fwd Seg Size Min','FWD Init Win Bytes','Packet Length Min',
        'Protocol','SYN Flag Count','Fwd Header Length']

train_df = pd.DataFrame(X_train, columns=feature_names)
train_df['y'] = y_train

print("TRAIN median by class")
print(train_df.groupby('y')[cols].median().T)

TRAIN median by class


KeyError: "Columns not found: 'Fwd Seg Size Min', 'Packet Length Min', 'Protocol', 'FWD Init Win Bytes'"